## HLS reconciliation for Sentinel S30 granules
Before running the notebook, please ensure the following python packages are installed. <br>
- Create a python environment <br>
python3 -m venv hls_recon
source hls_recon/bin/activate
- Install the following packages using pip or conda. <br> If you prefer having the following packages in a single file, save the packages in the requirements.txt, and then run [pip install -r requirements.txt] <br>
pip install jupyterlab notebook polars pandas requests tqdm pysolar mgrs 
- Register the environment as Jupyter kernel and run the following command <br>
pip install ipykernel
python -m ipykernel install --user --name hls_recon --display-name "Python (hls_recon)"
Launch Jupyter lab using [jupyter lab]

For the workflow, refer to the HLS ticket flowchart https://github.com/NASA-IMPACT/hls_development/issues/402

### Step 1: Reading the S2 downloader for a specfic date
This step will extract the granules from the S2 downloader for a specific date (say: 20250320). 
- Create a new directory, say S2_out and it should contain all .csv files +/- 2 days since the current date. You need to make that the file header of the .csv file are the same. It is suggested to move any other irrelavant files from the folder.
- We need to ensure that the HLS granules for a specific DATE are captured in the search criteria. 

In [2]:
from pathlib import Path
import polars as pl

# Set path or folder directory

input_path = Path("ESA_monthly_summaries/20250320/S2_out")
out_file = input_path / "S2_combined_filtered_20250320.csv"

# Search for date fielf in column 2 (Granule_Name)
pattern = "20250320"
col_index = 1  # 2nd column
all_results = []

files = sorted(input_path.glob("*.csv"))
print(f"Found {len(files)} csv files")

for filepath in files:
    try:
        df = pl.read_csv(
            filepath,
            separator="\t",
            has_header=False,
            ignore_errors=True
        )

        if df.width <= col_index:
            print(f"Skipping {filepath.name}: not enough columns")
            continue

        result = df.filter(
            pl.col(df.columns[col_index]).cast(pl.Utf8).str.contains(pattern)
        )

        if result.height > 0:
            # optional: add source filename as last column
            result = result.with_columns(
                pl.lit(filepath.name).alias("source_file")
            )
            all_results.append(result)

        print(f"{filepath.name}: {result.height} matches")

    except Exception as e:
        print(f"Error reading {filepath.name}: {e}")

if all_results:
    combined = pl.concat(all_results, how="vertical_relaxed")
    combined.write_csv(out_file, separator="\t", include_header=False)
    print(f"\nSaved {combined.height} total matched rows to {out_file}")
else:
    print("\nNo matches found in any file.")

Found 5 csv files
output_s2_downloader_db_2025-03-18.csv: 0 matches
output_s2_downloader_db_2025-03-19.csv: 0 matches
output_s2_downloader_db_2025-03-20.csv: 10914 matches
output_s2_downloader_db_2025-03-21.csv: 1142 matches
output_s2_downloader_db_2025-03-22.csv: 0 matches

Saved 12056 total matched rows to ESA_monthly_summaries/20250320/S2_out/S2_combined_filtered_20250320.csv


### Adding headers to the downloader file 

- The csv file from S2 downloader does not have a header, so it is recommended to add an header prior to the granule comparison.
- Header information include UUID, Granule_Name, Tile_ID, Size_Bytes,
    Sensing_Time, Creation_Time, Ingestion_Time, Download_URL,
    Checksum, Is_Land, Exit_Code, SZA_Flag, source_file

In [4]:
from pathlib import Path
import polars as pl

# File path
BASE_PATH = Path("./ESA_monthly_summaries/20250320/S2_out")
input_file = BASE_PATH / "S2_combined_filtered_20250320.csv"
output_file = BASE_PATH / "S2_combined_20250320.csv"

# Headers for 13 columns
new_headers = ['UUID', 'Granule_Name', 'Tile_ID', 'Size_Bytes',
    'Sensing_Time','Creation_Time','Ingestion_Time','Download_URL',
    'Checksum','Is_Land','Exit_Code','SZA_Flag','source_file'
]

# Reading the input file .csv file with tab
df = pl.read_csv(
    input_file,
    separator="\t",     
    has_header=False,
    ignore_errors=True
)

print("Columns detected:", df.width)

# Assign headers
if df.width == len(new_headers):
    df.columns = new_headers
else:
    print("Column mismatch. Something is wrong")
    print("Detected:", df.width)
    print("Expected:", len(new_headers))

# Save the output file
df.write_csv(output_file, separator=",")
print(f"Output file written to {output_file}")

Columns detected: 13
Output file written to ESA_monthly_summaries/20250320/S2_out/S2_combined_20250320.csv


### Step 2: Retrieve granules from NASA Earthdata Search (CMR query)

First, we will run a quick search to find the granules for HLS L30 and S30, and then retrieve the metadata elements using CMR Search.

In [5]:
from process_esa_S2_v1 import get_hls_granule_count

target_date = "2025-03-20"
count = get_hls_granule_count(target_date)

---- Querying CMR for HLS Granules on 2025-03-20 ----

HLS S30 Granule: 8,422 granules
HLS L30 Granule: 5,320 granules
Total granules: 13,742


### Get the metadata for the granule search based on the date input

In [6]:
import requests
import pandas as pd
from process_esa_S2_v1 import get_hls_metadata

TARGET_DATE = "2025-03-20"          # here we give the date as string to match CMR 
OUTPUT_CSV = f"ESA_monthly_summaries/20250320/HLS_S30_NASA_CMR_Metadata_{TARGET_DATE}.csv"

results = get_hls_metadata(TARGET_DATE)

if results:
    df = pd.DataFrame(results)
    
    cols = ["Granule_ID", "Product_URI", "Insert_Time", "Last_Update", 
            "Beginning_DateTime", "Ending_DateTime", "Cloud_Cover", "Mean_Sun_Zenith_Angle"]
    
    # Reorder if columns exist
    existing_cols = [c for c in cols if c in df.columns]
    df = df[existing_cols]
    df.to_csv(OUTPUT_CSV, index=False)

    print(f" Success! Extracted {len(df)} records.")
    print(f" Saved to: {OUTPUT_CSV}")
    #print(df.head(1))
else:
    print("No granules found.")

Retrieving the CMR metadata for 2025-03-20...
   PAGE 1: Found 2000 granules...
   PAGE 2: Found 2000 granules...
   PAGE 3: Found 2000 granules...
   PAGE 4: Found 2000 granules...
   PAGE 5: Found 2000 granules...
   PAGE 6: Found 2000 granules...
   PAGE 7: Found 1742 granules...
 Success! Extracted 13742 records.
 Saved to: ESA_monthly_summaries/20250320/HLS_S30_NASA_CMR_Metadata_2025-03-20.csv


#### We will run a quick check to find rows with 'PRODUCT_URI' 
- Thoughts - What if the 'PRODUCT_URI' is missed out accidentally during the metadata processing in the CMR !!!
- There is also possibility of having a twin granule, which we will resolve it later. As of now, we will report the number of HLS granules with twin Sentinel Product_URI

In [16]:
import polars as pl

file_path = "ESA_monthly_summaries/20250320/HLS_S30_NASA_CMR_Metadata_2025-03-20.csv"

# Read CSV
df = pl.read_csv(file_path, ignore_errors=True)

# -----------------------------
# Normalize column (important)
# -----------------------------
df = df.with_columns(
    pl.col("Product_URI")
      .cast(pl.Utf8)
      .str.strip_chars()
      .alias("Product_URI")
)

# Count rows
total_rows = df.height

rows_with_value = df.filter(
    pl.col("Product_URI").is_not_null() &
    (pl.col("Product_URI") != "")
).height

rows_without_value = total_rows - rows_with_value

# Condition: contains '+' for twin granules
# -----------------------------
rows_with_plus = df.filter(
    pl.col("Product_URI").is_not_null() &
    pl.col("Product_URI").str.contains(r"\+")
).height

# -----------------------------
# Print summary
# -----------------------------
print("ESA Product_URI information")
print(f"Total HLS granules from NASA CMR:  {total_rows}")
print(f"HLS Granules with Product_URI   :  {rows_with_value}")
print(f"HLS Granules without Product_URI:  {rows_without_value}")
print(f"HLS Granules with twin granules :  {rows_with_plus}")

ESA Product_URI information
Total HLS granules from NASA CMR:  13742
HLS Granules with Product_URI   :  8422
HLS Granules without Product_URI:  5320
HLS Granules with twin granules :  189


### Step 3: Compare the granules from S2 downloader and NASA CMR 
- Once we know the granule count from NASA Earthdata search (CMR) and S2 downloader, we can identify the missing granules that are not processed by the HLS pipeline. 
- The code will return 3 files a) matched granules list, b) unmatched or not processed by HLS workflow, and (c) double granules or missing granule ID (due to missing metadata in the CMR). 
- We are interested in the file (unmatched or not processed granules list) for the next step. 

In [24]:
import polars as pl

# --- Set file path ---
file1_csv = "ESA_monthly_summaries/20250320/HLS_S30_NASA_CMR_Metadata_2025-03-20.csv"
file2_csv = "ESA_monthly_summaries/20250320/S2_combined_20250320.csv"

output_matched = "ESA_monthly_summaries/20250320/HLS_Combined_Metadata_Matched.csv"
output_unmatched_file2 = "ESA_monthly_summaries/20250320/HLS_unmatched_not_processed.csv"
output_unmatched_file1 = "ESA_monthly_summaries/20250320/HLS_twin_granule.csv"

print("Reading files...")

# 1) Read both CSV files
df_file1 = pl.read_csv(file1_csv)
df_file2 = pl.read_csv(file2_csv)

# --- Initial granules list from both files ---
print("--- Initial Row Count Statistics ---")
print(f"HLS granules from NASA CMR: {df_file1.height:,}")
print(f"HLS granules from S2 downloader: {df_file2.height:,}")
print("-" * 38 + "\n")

# --- Clean whitespace to prevent false mismatches ---
df_file1 = df_file1.with_columns(pl.col("Product_URI").cast(pl.Utf8).str.strip_chars())
df_file2 = df_file2.with_columns(pl.col("Granule_Name").cast(pl.Utf8).str.strip_chars())

print("Performing joins...")

# Matched - Inner Join
df_matched = df_file2.join(
    df_file1,
    left_on="Granule_Name",
    right_on="Product_URI",
    how="inner"
)

# UNMATCHED rows in file2 (Anti Join)
df_unmatched_file2 = df_file2.join(
    df_file1,
    left_on="Granule_Name",
    right_on="Product_URI",
    how="anti"
)

# UNMATCHED rows in file1 (Anti Join)
df_unmatched_file1 = df_file1.join(
    df_file2,
    left_on="Product_URI",
    right_on="Granule_Name",
    how="anti"
)

print("Saving files...")
df_matched.write_csv(output_matched)

if df_unmatched_file2.height > 0:
    df_unmatched_file2.write_csv(output_unmatched_file2)

if df_unmatched_file1.height > 0:
    df_unmatched_file1.write_csv(output_unmatched_file1)

# --- Final comparison summary ---
print("----Comparison of HLS granules from S2 downloader and NASA CMR---")
print(f"Matched granules (NASA CMR vs S2 downloader): {df_matched.height:,}  -> {output_matched}")
print(f"Unmatched granules (Not processed in the HLS): {df_unmatched_file2.height:,}" +
      (f" -> {output_unmatched_file2}" if df_unmatched_file2.height > 0 else " (None)"))
print(f"Unmatched granules (Errors): {df_unmatched_file1.height:,}" +
      (f"                   -> {output_unmatched_file1}" if df_unmatched_file1.height > 0 else " (None)"))

Reading files...
--- Initial Row Count Statistics ---
HLS granules from NASA CMR: 13,742
HLS granules from S2 downloader: 12,056
--------------------------------------

Performing joins...
Saving files...
----Comparison of HLS granules from S2 downloader and NASA CMR---
Matched granules (NASA CMR vs S2 downloader): 8,233  -> ESA_monthly_summaries/20250320/HLS_Combined_Metadata_Matched.csv
Unmatched granules (Not processed in the HLS): 3,823 -> ESA_monthly_summaries/20250320/HLS_unmatched_not_processed.csv
Unmatched granules (Errors): 5,509                   -> ESA_monthly_summaries/20250320/HLS_twin_granule.csv


### Step 4: Retrieve the cloud cover from ESA Copernicus catalog 

- In the HLS unmatched granules, we do NOT have the cloud cover and solar zenith angle (SZA). 
- We get this information from ESA Copernicus catalog based on the 'Product_URI' (Granule_name). 
- Since large information is processed using OData, we use multithreading option for efficient query search. 

In [25]:
from process_esa_S2_v1 import enrich_csv_fast

input_file1 = "ESA_monthly_summaries/20250320/HLS_unmatched_not_processed.csv"
output_file1 = "ESA_monthly_summaries/20250320/HLS_unmatched_not_processed_ESA_CC.csv"
enrich_csv_fast(input_file1, output_file1, max_workers=15)

Reading the file ESA_monthly_summaries/20250320/HLS_unmatched_not_processed.csv...
Number of 3823 unique granules to query.
Working on 15 concurrent threads...



Merging metadata back into the main dataset...
Saved metadata from ESA with 3823 rows to ESA_monthly_summaries/20250320/HLS_unmatched_not_processed_ESA_CC.csv


### Step 5: Compute the solar zenith angle (SZA) using pysolar

We compute the SZA using pysolar package https://pysolar.readthedocs.io/en/latest/. Since it was difficult to retrieve the SZA or illumination angle from ESA Copernicus catalog, we compute it via pysolar as of now. Will switch to a better method later. 

In [26]:
import re
from datetime import datetime, timezone
from pathlib import Path
import polars as pl
from process_esa_S2_v1 import compute_sza
import mgrs

INPUT_FILE = Path("ESA_monthly_summaries/20250320/HLS_unmatched_not_processed_ESA_CC.csv")
OUTPUT_FILE = Path("ESA_monthly_summaries/20250320/HLS_not_processed_CC_SZA.csv")

# Helpers
_TIME_RE = re.compile(r"_(\d{8}T\d{6})_")
_TILE_RE = re.compile(r"_T(\d{2}[A-Z]{3})_")

_mgrs = mgrs.MGRS()
_tile_cache: dict[str, tuple[float, float]] = {}  # tile_id -> (lat, lon)

df = pl.read_csv(INPUT_FILE, ignore_errors=True)

# Ensure columns exist
needed = {"Granule_Name", "Tile_ID"}
missing = needed - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found columns: {df.columns}")

# Compute SZA (Python UDF; fine for ~10k–100k rows)
df_out = df.with_columns(
    pl.struct(["Granule_Name", "Tile_ID"]).map_elements(
        lambda row: compute_sza(row["Granule_Name"], row["Tile_ID"]),
        return_dtype=pl.Float64,
    ).alias("Solar_Zenith_Angle")
)

df_out.write_csv(OUTPUT_FILE)
print(f"Computed SZA for {df_out.height} rows and saved to: {OUTPUT_FILE}")

# Optional check for null
print("SZA nulls:", df_out.select(pl.col("Solar_Zenith_Angle").is_null().sum()).item())

Computed SZA for 3823 rows and saved to: ESA_monthly_summaries/20250320/HLS_not_processed_CC_SZA.csv
SZA nulls: 0


### Step 6: Final count of granules meeting the CC and SZA conditions

In [27]:
import polars as pl
from pathlib import Path

INPUT_FILE = Path("ESA_monthly_summaries/20250320/HLS_not_processed_CC_SZA.csv")

CLOUD_THRESH = 95.0
SZA_THRESH = 76.0

df = pl.read_csv(INPUT_FILE, ignore_errors=True)

# --- ensure numeric (in case they were read as strings) ---
df = df.with_columns([
    pl.col("ESA_cloud").cast(pl.Float64, strict=False),
    pl.col("Solar_Zenith_Angle").cast(pl.Float64, strict=False),
])

total = df.height

# Decide how to treat missing values:
# Keep nulls (i.e., don't eliminate if missing)
cloud_ok = (pl.col("ESA_cloud").is_null()) | (pl.col("ESA_cloud") <= CLOUD_THRESH)
sza_ok   = (pl.col("Solar_Zenith_Angle").is_null()) | (pl.col("Solar_Zenith_Angle") <= SZA_THRESH)

# Filtered subsets (these are the rows you KEEP)
df_cloud = df.filter(cloud_ok)
df_sza   = df.filter(sza_ok)
df_both  = df.filter(cloud_ok & sza_ok)

# Eliminated counts
elim_cloud = df.filter(pl.col("ESA_cloud") > CLOUD_THRESH).height
elim_sza   = df.filter(pl.col("Solar_Zenith_Angle") > SZA_THRESH).height
elim_both  = df.filter((pl.col("ESA_cloud") > CLOUD_THRESH) | (pl.col("Solar_Zenith_Angle") > SZA_THRESH)).height
elim_and  = df.filter((pl.col("ESA_cloud") > CLOUD_THRESH) & (pl.col("Solar_Zenith_Angle") > SZA_THRESH)).height

print("--- Final HLS Summary for missing granules ---")
print(f"Total rows: {total:,}")
print("\n Cloud cover filter")
print(f"  Threshold: ESA_cloud > {CLOUD_THRESH}")
print(f"  Eliminated granule count: {elim_cloud:,}")
print(f"  Remaining granules (cloud ok): {df_cloud.height:,}")

print("\n SZA filter")
print(f"  Threshold: Solar_Zenith_Angle > {SZA_THRESH}")
print(f"  Eliminated granule count: {elim_sza:,}")
print(f"  Remaining granules(SZA ok): {df_sza.height:,}")

print("\n Combined filter (either condition)")
print(f"  Eliminated (cloud > {CLOUD_THRESH} OR SZA > {SZA_THRESH}): {elim_both:,}")
print(f"  Remaining (cloud ok AND SZA ok): {df_both.height:,}")
print(f"  Eliminated (cloud > {CLOUD_THRESH} AND SZA > {SZA_THRESH}): {elim_and:,}")

# Optional: report how many have missing values
null_cloud = df.select(pl.col("ESA_cloud").is_null().sum()).item()
null_sza = df.select(pl.col("Solar_Zenith_Angle").is_null().sum()).item()
print("\n Missing values")
print(f"  ESA_cloud nulls: {null_cloud:,}")
print(f"  Solar_Zenith_Angle nulls: {null_sza:,}")

--- Final HLS Summary for missing granules ---
Total rows: 3,823

 Cloud cover filter
  Threshold: ESA_cloud > 95.0
  Eliminated granule count: 2,090
  Remaining granules (cloud ok): 1,733

 SZA filter
  Threshold: Solar_Zenith_Angle > 76.0
  Eliminated granule count: 883
  Remaining granules(SZA ok): 2,940

 Combined filter (either condition eliminates)
  Eliminated (cloud > 95.0 OR SZA > 76.0): 2,746
  Remaining (cloud ok AND SZA ok): 1,077
  Eliminated (cloud > 95.0 AND SZA > 76.0): 227

 Missing values
  ESA_cloud nulls: 12
  Solar_Zenith_Angle nulls: 0
